In [1]:
# Cell 1: Environment steup, loading data
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt 
import torch
import torch.nn as nn


In [2]:
# Cell 2: LSTM Sequence generation (sliding windows)
def create_sequences(df_ticker, seq_length):
    features = ['Close', 'Volume', 'SMA_20', 'RSI_14']

    target = 'Close'

    X, y, dates = [], [], []

    for i in range(len(df_ticker) - seq_length):
        seq_x = df_ticker[features].iloc[i : i + seq_length].values 
        seq_y = df_ticker[target].iloc[i + seq_length]
        seq_date = df_ticker['Trading_Date'].iloc[i + seq_length]
        

        X.append(seq_x)
        y.append(seq_y)
        dates.append(seq_date)

    return np.array(X), np.array(y), np.array(dates)


In [3]:
# Cell 3: Building the deep learning model (LSTM)

class StockPredictorLSTM(nn.Module):
    def __init__(self, input_size, hidden_size, num_layers, output_size=1, dropout_rate=0.2):
        super(StockPredictorLSTM, self).__init__()

        # Core LSTM Layer
        self.lstm = nn.LSTM(
            input_size=input_size,
            hidden_size=hidden_size,
            num_layers=num_layers,
            batch_first=True,
            dropout=dropout_rate if num_layers > 1 else 0
        )
        self.dropout = nn.Dropout(dropout_rate)

        self.fc1 = nn.Linear(hidden_size, 32)
        self.relu = nn.ReLU()

        # Output prediction layer
        self.fc2 = nn.Linear(32, output_size)

    def forward(self, x):
        lstm_out, _ = self.lstm(x)
        last_timestep_out = lstm_out[:, -1, :]

        out = self.dropout(last_timestep_out)
        out = self.fc1(out)
        out = self.relu(out)
        out = self.fc2(out)

        return out


# Init model
INPUT_SIZE = 4
HIDDEN_SIZE = 256  
NUM_LAYERS = 1     
OUTPUT_SIZE = 1   
model = StockPredictorLSTM(
    input_size=INPUT_SIZE, 
    hidden_size=HIDDEN_SIZE, 
    num_layers=NUM_LAYERS, 
    output_size=OUTPUT_SIZE
)
print(model)



StockPredictorLSTM(
  (lstm): LSTM(4, 256, batch_first=True)
  (dropout): Dropout(p=0.2, inplace=False)
  (fc1): Linear(in_features=256, out_features=32, bias=True)
  (relu): ReLU()
  (fc2): Linear(in_features=32, out_features=1, bias=True)
)


In [4]:
# Cell 4: Standardized Triple Loop (V1 LSTM)
import os
from sklearn.preprocessing import MinMaxScaler
from sklearn.metrics import mean_absolute_error, r2_score
from torch.utils.data import TensorDataset, DataLoader

# --- CONFIGS ---
TICKERS = ['MSFT', 'DIS', 'WMT'] 
SEQ_LENGTHS = [5, 10, 20] 
NUM_RUNS = 10
EPOCHS = 100
BATCH_SIZE = 128
LEARNING_RATE = 0.001

BASE_MODEL_DIR = "models/v1_PriceOnly"
BASE_RESULTS_DIR = "results/trainV1_PriceOnly"
os.makedirs(BASE_RESULTS_DIR, exist_ok=True)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

for seq_len in SEQ_LENGTHS:
    print(f"\n" + "#"*60 + f"\nSTARTING V1 EXPERIMENTS | SEQUENCE LENGTH: {seq_len}\n" + "#"*60)
    SEQ_RESULTS = []
    
    for ticker in TICKERS:
        print(f"\n>>> Processing: {ticker} (Seq: {seq_len})")
        
        file_path = f"datasets/{ticker}/{ticker}_DatasetV1.csv"
        if not os.path.exists(file_path):
            print(f"Error: {file_path} not found.")
            continue
            
        ticker_df = pd.read_csv(file_path)
        ticker_df['Trading_Date'] = pd.to_datetime(ticker_df['Trading_Date'])
        
        # Split by Year
        train_raw = ticker_df[ticker_df['Trading_Date'].dt.year <= 2022].copy()
        test_raw = ticker_df[ticker_df['Trading_Date'].dt.year == 2023].copy()
        
        # Scale: Fit on Train Only
        scaler = MinMaxScaler()
        cols = ['Close', 'Volume', 'SMA_20', 'RSI_14']
        train_raw[cols] = scaler.fit_transform(train_raw[cols])
        test_raw[cols] = scaler.transform(test_raw[cols])
        
        # Generate Sequences
        X_train, y_train, _ = create_sequences(train_raw, seq_len)
        X_test, y_test, _ = create_sequences(test_raw, seq_len)
        
        X_train_t = torch.tensor(X_train, dtype=torch.float32).to(device)
        y_train_t = torch.tensor(y_train, dtype=torch.float32).unsqueeze(1).to(device)
        X_test_t = torch.tensor(X_test, dtype=torch.float32).to(device)

        for run in range(1, NUM_RUNS + 1):
            # Single-layer LSTM, 256 units (V1 Architecture)
            model = StockPredictorLSTM(input_size=4, hidden_size=256, num_layers=1).to(device)
            optimizer = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE)
            criterion = nn.MSELoss()
            
            train_loader = DataLoader(TensorDataset(X_train_t, y_train_t), batch_size=BATCH_SIZE, shuffle=True)
            
            model.train()
            for epoch in range(EPOCHS):
                for xb, yb in train_loader:
                    optimizer.zero_grad()
                    criterion(model(xb), yb).backward()
                    optimizer.step()
            
            model.eval()
            with torch.no_grad():
                preds = model(X_test_t).cpu().numpy().flatten()
            
            r2 = r2_score(y_test, preds)
            mae = mean_absolute_error(y_test, preds)
            
            # Store Metrics
            SEQ_RESULTS.append({'Ticker': ticker, 'Run': run, 'Seq': seq_len, 'R2': r2, 'MAE': mae})
            
            # Save weights (.pth)
            save_path = os.path.join(BASE_MODEL_DIR, ticker)
            os.makedirs(save_path, exist_ok=True)
            torch.save(model.state_dict(), f"{save_path}/v1PriceOnly_run_{run}_seq{seq_len}.pth")
            
            print(f"  Run {run:02d}: R2={r2:.4f}")

    # After each sequence length, save a CSV report
    results_df = pd.DataFrame(SEQ_RESULTS)
    csv_path = os.path.join(BASE_RESULTS_DIR, f"v1PriceOnly_stability_results_{seq_len}.csv")
    results_df.to_csv(csv_path, index=False)
    
    print(f"\n[DONE] Seq {seq_len} results saved to: {csv_path}")
    display(results_df.groupby('Ticker')[['R2', 'MAE']].agg(['mean', 'std']))



############################################################
STARTING V1 EXPERIMENTS | SEQUENCE LENGTH: 5
############################################################

>>> Processing: MSFT (Seq: 5)
  Run 01: R2=0.9800
  Run 02: R2=0.9216
  Run 03: R2=0.9812
  Run 04: R2=0.9385
  Run 05: R2=0.9806
  Run 06: R2=0.9664
  Run 07: R2=0.9792
  Run 08: R2=0.9550
  Run 09: R2=0.9800
  Run 10: R2=0.9811

>>> Processing: DIS (Seq: 5)
  Run 01: R2=0.9383
  Run 02: R2=0.9169
  Run 03: R2=0.8872
  Run 04: R2=0.8916
  Run 05: R2=0.8895
  Run 06: R2=0.9047
  Run 07: R2=0.8234
  Run 08: R2=0.9079
  Run 09: R2=0.8593
  Run 10: R2=0.9220

>>> Processing: WMT (Seq: 5)
  Run 01: R2=0.9382
  Run 02: R2=0.9416
  Run 03: R2=0.8619
  Run 04: R2=0.9442
  Run 05: R2=0.7872
  Run 06: R2=0.9175
  Run 07: R2=0.9019
  Run 08: R2=0.9331
  Run 09: R2=0.8187
  Run 10: R2=0.8823

[DONE] Seq 5 results saved to: results/trainV1_PriceOnly\v1PriceOnly_stability_results_5.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.894079  0.033065  0.017380  0.002959
MSFT    0.966355  0.021320  0.023309  0.008117
WMT     0.892648  0.054873  0.030332  0.009524


############################################################
STARTING V1 EXPERIMENTS | SEQUENCE LENGTH: 10
############################################################

>>> Processing: MSFT (Seq: 10)
  Run 01: R2=0.9414
  Run 02: R2=0.9486
  Run 03: R2=0.9725
  Run 04: R2=0.9756
  Run 05: R2=0.9617
  Run 06: R2=0.9429
  Run 07: R2=0.9216
  Run 08: R2=0.9764
  Run 09: R2=0.9691
  Run 10: R2=0.9756

>>> Processing: DIS (Seq: 10)
  Run 01: R2=0.8601
  Run 02: R2=0.9224
  Run 03: R2=0.8476
  Run 04: R2=0.8782
  Run 05: R2=0.8820
  Run 06: R2=0.9135
  Run 07: R2=0.9246
  Run 08: R2=0.8779
  Run 09: R2=0.8820
  Run 10: R2=0.8932

>>> Processing: WMT (Seq: 10)
  Run 01: R2=0.7675
  Run 02: R2=0.7302
  Run 03: R2=0.8807
  Run 04: R2=0.8081
  Run 05: R2=0.8696
  Run 06: R2=0.8511
  Run 07: R2=0.8022
  Run 08: R2=0.7768
  Run 09: R2=0.6240
  Run 10: R2=0.7658

[DONE] Seq 10 results saved to: results/trainV1_PriceOnly\v1PriceOnly_stability_results_10.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.888141  0.025578  0.018130  0.002154
MSFT    0.958526  0.018906  0.025525  0.006599
WMT     0.787601  0.075369  0.045335  0.008877


############################################################
STARTING V1 EXPERIMENTS | SEQUENCE LENGTH: 20
############################################################

>>> Processing: MSFT (Seq: 20)
  Run 01: R2=0.9624
  Run 02: R2=0.8434
  Run 03: R2=0.9582
  Run 04: R2=0.9405
  Run 05: R2=0.9777
  Run 06: R2=0.9334
  Run 07: R2=0.8954
  Run 08: R2=0.9559
  Run 09: R2=0.9441
  Run 10: R2=0.8889

>>> Processing: DIS (Seq: 20)
  Run 01: R2=0.8432
  Run 02: R2=0.8288
  Run 03: R2=0.8085
  Run 04: R2=0.9115
  Run 05: R2=0.9212
  Run 06: R2=0.8889
  Run 07: R2=0.8945
  Run 08: R2=0.9176
  Run 09: R2=0.8523
  Run 10: R2=0.7934

>>> Processing: WMT (Seq: 20)
  Run 01: R2=0.8659
  Run 02: R2=0.9269
  Run 03: R2=0.8846
  Run 04: R2=0.6542
  Run 05: R2=0.9209
  Run 06: R2=0.8990
  Run 07: R2=0.6336
  Run 08: R2=0.8436
  Run 09: R2=0.8013
  Run 10: R2=0.9166

[DONE] Seq 20 results saved to: results/trainV1_PriceOnly\v1PriceOnly_stability_results_20.csv


R2                 MAE          
            mean       std      mean       std
Ticker                                        
DIS     0.865988  0.046874  0.018824  0.003727
MSFT    0.929987  0.041499  0.031107  0.010658
WMT     0.834666  0.107733  0.035538  0.012211